In [14]:
import os
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.utils import shuffle
from scipy.signal import resample
import matplotlib.pyplot as plt
import json
import warnings
import mne
from mne.preprocessing import ICA
warnings.filterwarnings("ignore")
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [15]:
SAMPLE_RATE = 200  # fs
# SAMPLE_LEN = 1.0   # sample seconds
# OVERLAPPING = 0.8  # overlapping seconds
sub_folder_path = str(SAMPLE_RATE) + 'Hz'
sub_folder_path

'200Hz'

In [16]:
## Load participants.tsv file
# root dir
root = "SCPD/"
participants_path = os.path.join(root, "participants.tsv")
participants = pd.read_csv(participants_path, sep="\t")
participants


,participant_id,Original_ID,Group,sess1_Med,sess2_Med,sex,age
0,sub-001,8010,CTL,NaN,no s2,Female,61
1,sub-002,801,PD,ON,OFF,Female,60
2,sub-003,802,PD,OFF,ON,Male,75
3,sub-004,803,PD,OFF,ON,Female,76
4,sub-005,804,PD,ON,OFF,Male,75
5,sub-006,805,PD,ON,OFF,Male,79
6,sub-007,8060,CTL,NaN,no s2,Female,83
7,sub-008,806,PD,OFF,ON,Female,79
8,sub-009,8070,CTL,NaN,no s2,Female,67
9,sub-010,807,PD,OFF,ON,Female,72


In [17]:
# Test for bad channels, sampling freq and shape
"""bad_channel_list, sampling_freq_list, data_shape_list = [], [], []
for sub in os.listdir(root):
    if sub.startswith("sub-"):   # Confirm it is a subject directory
        sub_path = os.path.join(root, sub)
        for ses in os.listdir(sub_path):
            ses_path = os.path.join(sub_path, ses, "eeg")
            if os.path.exists(ses_path):
                for file in os.listdir(ses_path):
                    if file.endswith(".set"):
                        file_path = os.path.join(ses_path, file)
                        print(f"Reading in progress: {file_path}")
                        
                        try:
                            # Try loading raw first
                            raw = mne.io.read_raw_eeglab(file_path, preload=True)
                            bad_channel_list.append(raw.info['bads'])
                            sampling_freq_list.append(raw.info['sfreq'])
                            data_shape_list.append(raw.get_data().shape)
                        except Exception as e1:
                            print(f"  -> If raw fails, try epochs: {e1}")
                            try:
                                epochs = mne.read_epochs_eeglab(file_path)
                                epochs.load_data()
                                bad_channel_list.append(epochs.info['bads'])
                                sampling_freq_list.append(epochs.info['sfreq'])
                                data_shape_list.append(epochs.get_data().shape)
                            except Exception as e2:
                                print(f"  -> Failed to load epochs as well: {e2}")"""

'bad_channel_list, sampling_freq_list, data_shape_list = [], [], []\nfor sub in os.listdir(root):\n    if sub.startswith("sub-"):   # Confirm it is a subject directory\n        sub_path = os.path.join(root, sub)\n        for ses in os.listdir(sub_path):\n            ses_path = os.path.join(sub_path, ses, "eeg")\n            if os.path.exists(ses_path):\n                for file in os.listdir(ses_path):\n                    if file.endswith(".set"):\n                        file_path = os.path.join(ses_path, file)\n                        print(f"Reading in progress: {file_path}")\n                        \n                        try:\n                            # Try loading raw first\n                            raw = mne.io.read_raw_eeglab(file_path, preload=True)\n                            bad_channel_list.append(raw.info[\'bads\'])\n                            sampling_freq_list.append(raw.info[\'sfreq\'])\n                            data_shape_list.append(raw.get_data().shape

In [18]:
"""from collections import Counter

print(bad_channel_list)
print(data_shape_list[0])
print("Channel number counter:", Counter(i[0] for i in data_shape_list))
print("Sampling rate counter:", Counter(sampling_freq_list))"""

'from collections import Counter\n\nprint(bad_channel_list)\nprint(data_shape_list[0])\nprint("Channel number counter:", Counter(i[0] for i in data_shape_list))\nprint("Sampling rate counter:", Counter(sampling_freq_list))'

In [19]:
"""common_channels = []
for sub in os.listdir(root):
    if sub.startswith("sub-"):   # Only process subject directories
        sub_path = os.path.join(root, sub)
        for ses in os.listdir(sub_path):  # Iterate over ses-xx directories
            ses_path = os.path.join(sub_path, ses, "eeg")
            if os.path.exists(ses_path):
                for file in os.listdir(ses_path):
                    if file.endswith(".set"):
                        file_path = os.path.join(ses_path, file)
                        print(f"Reading in progress: {file_path}")
                        
                        # Try loading raw data first, fallback to epochs if raw fails
                        try:
                            raw = mne.io.read_raw_eeglab(file_path, preload=True)
                            current_channels = set(raw.info['ch_names'])
                        except Exception as e1:
                            print(f"  -> Raw read failed, fallback to epochs: {e1}")
                            try:
                                epochs = mne.read_epochs_eeglab(file_path)
                                epochs.load_data()
                                current_channels = set(epochs.info['ch_names'])
                            except Exception as e2:
                                print(f"  -> Failed to read epochs as well: {e2}")
                                continue  # Skip this file
                        
                        # Find common channels
                        if not common_channels:
                            common_channels = current_channels
                        else:
                            common_channels &= current_channels

common_channels = list(common_channels)
print(f"Common channels ({len(common_channels)}):", common_channels)"""

'common_channels = []\nfor sub in os.listdir(root):\n    if sub.startswith("sub-"):   # Only process subject directories\n        sub_path = os.path.join(root, sub)\n        for ses in os.listdir(sub_path):  # Iterate over ses-xx directories\n            ses_path = os.path.join(sub_path, ses, "eeg")\n            if os.path.exists(ses_path):\n                for file in os.listdir(ses_path):\n                    if file.endswith(".set"):\n                        file_path = os.path.join(ses_path, file)\n                        print(f"Reading in progress: {file_path}")\n                        \n                        # Try loading raw data first, fallback to epochs if raw fails\n                        try:\n                            raw = mne.io.read_raw_eeglab(file_path, preload=True)\n                            current_channels = set(raw.info[\'ch_names\'])\n                        except Exception as e1:\n                            print(f"  -> Raw read failed, fallback to epo

In [20]:
common_channels = ['FC2', 'P6', 'FT8', 'TP9', 'FC5', 'CP1', 'F4', 'AF4', 'O1', 'C4', 'CP6', 'P7', 'AF8', 'PO8', 'AF7', 'TP7', 'F7', 'Pz', 'FC4', 'CP5', 'Fp2', 'P1', 'FT9', 'POz', 'Fp1', 'TP10', 'CP3', 'CP2', 'F8', 'PO4', 'PO7', 'P8', 'VEOG', 'P5', 'AFz', 'T8', 'Cz', 'FC1', 'F1', 'PO3', 'C2', 'P2', 'X', 'Fz', 'Oz', 'C3', 'P4', 'FC6', 'P3', 'FC3', 'C1', 'FT10', 'C5', 'C6', 'F6', 'FT7', 'O2', 'CP4', 'F3', 'F2', 'F5', 'Y', 'AF3', 'TP8', 'Z', 'T7', 'FCz']

In [21]:
# remove unreliable electrodes FT9, FT10, TP9, TP10 and non-EEG channels X, Y, Z, VEOG
common_channels = [ch for ch in common_channels if ch not in ['FT9', 'FT10', 'TP9', 'TP10', 'X', 'Y', 'Z', 'VEOG']]
print(common_channels)
print("Common channels number: ", len(common_channels))

['FC2', 'P6', 'FT8', 'FC5', 'CP1', 'F4', 'AF4', 'O1', 'C4', 'CP6', 'P7', 'AF8', 'PO8', 'AF7', 'TP7', 'F7', 'Pz', 'FC4', 'CP5', 'Fp2', 'P1', 'POz', 'Fp1', 'CP3', 'CP2', 'F8', 'PO4', 'PO7', 'P8', 'P5', 'AFz', 'T8', 'Cz', 'FC1', 'F1', 'PO3', 'C2', 'P2', 'Fz', 'Oz', 'C3', 'P4', 'FC6', 'P3', 'FC3', 'C1', 'C5', 'C6', 'F6', 'FT7', 'O2', 'CP4', 'F3', 'F2', 'F5', 'AF3', 'TP8', 'T7', 'FCz']
Common channels number:  59


In [22]:
def data_preprocessing(
    raw: mne.io.Raw,
    common_channels: list,
    sample_rate: int = 250,
    notch_freq: float = 60.0,
    l_freq: float = 0.1,
    h_freq: float = 20.0,
    do_bad_interp: bool = True,
    verbose: bool = True,
):
    """
    Simon Conflict Task EEG Preprocessing:
      1) Select common channels, remove unreliable ones (FT9, FT10, TP9, TP10), reorder
      2) Set Montage
      3) 60 Hz Notch filter
      4) Bandpass filter (0.1–20 Hz)
      5) Interpolate bad channels (if enabled)
      6) Re-reference to average
      7) ICA removal (fit on 1 Hz copy + ICLabel)
      8) Downsample to 250 Hz (if needed)
    """

    # 1. Remove unreliable electrodes
    keep = common_channels
    raw.pick_channels(keep)
    raw.reorder_channels(keep)
    if verbose:
        print(f"✔ Step 1: Picked & reordered channels {keep}")

    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage('standard_1020'))
    if verbose:
        print("✔ Step 2: Montage set to 'standard_1020'.")

    # 3. Notch filter (60 Hz)
    if notch_freq is not None:
        raw.notch_filter(freqs=[notch_freq], picks="eeg", verbose=False)
        if verbose:
            print(f"✔ Step 3: Notch filtered at {notch_freq} Hz")

    # 4. Bandpass filter (0.1–20 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Step 4: Band-pass filtered from {l_freq}–{h_freq} Hz")

    # 5. Interpolate bad channels
    if do_bad_interp and raw.info.get("bads"):
        raw.interpolate_bads(reset_bads=True, verbose=False)
        if verbose:
            print(f"✔ Step 5: Interpolated bad channels: {raw.info.get('bads', [])}")
    else:
        if verbose:
            print("ℹ Step 5: No bad channels to interpolate.")

    # 6. Re-reference to average
    raw.set_eeg_reference("average", verbose=False)
    if verbose:
        print("✔ Step 6: EEG re-referenced to average")

    # 7. ICA
    raw_for_ica = raw.copy().filter(l_freq=1.0, h_freq=None, picks="eeg", verbose=False)
    ica = ICA(n_components=None, method="fastica", random_state=97, max_iter="auto")

    ica.fit(raw_for_ica)

    excluded = []
    if label_components is not None:
        try:
            ic_labels = label_components(raw_for_ica, ica, method="iclabel")
            labels = ic_labels["labels"]
            probs = ic_labels["y_pred_proba"]
            thresholds = {
                "eye blink": 0.7,
                "muscle artifact": 0.6,
                "heart beat": 0.5,
                "line noise": 0.8,
                "channel noise": 0.9,
            }
            for i, lab in enumerate(labels):
                if lab in thresholds:
                    p = probs[i].max() if probs is not None else 1.0
                    if p >= thresholds[lab]:
                        excluded.append(i)
        except Exception as e:
            if verbose:
                print(f"⚠ ICLabel failed: {e}")
    else:
        if verbose:
            print("ℹ ICLabel not available; ICA fitted but no auto exclusion")

    if excluded:
        ica.exclude = sorted(set(excluded))
        raw = ica.apply(raw.copy())
        if verbose:
            print(f"✔ Step 7: ICA applied. Excluded components: {ica.exclude}")
    else:
        if verbose:
            print("ℹ Step 7: No ICA components excluded.")

    # 8. Downsample (if not already 250 Hz)
    if raw.info["sfreq"] != sample_rate:
        raw.resample(sample_rate, npad="auto", verbose=False)
        if verbose:
            print(f"✔ Step 8: Resampled to {sample_rate} Hz")

    return raw

In [23]:
import numpy as np
import pandas as pd
import mne


def epoch_and_make_xy(
    raw: mne.io.Raw,
    events_tsv_path: str,
    tmin: float = -0.5,
    tmax: float = 1.0,
    baseline=(-0.3, -0.2),
    task_id: int = 1,
    subject_id: int = 1,
    disease_id: int = 1,
):
    """
    Epoch Simon Conflict data using cue-lock strategy.

    Output:
        X: (n_trials, n_times, n_channels)
        y: (n_trials, 4) = [disease_id, stimulus_type, subject_id, task_id]

    stimulus_type:
        Training phase:
            0 = train_yellow_congruent
            1 = train_yellow_incongruent
            2 = train_blue_congruent
            3 = train_blue_incongruent

        Test phase:
            4 = test_AB
            5 = test_AC
            6 = test_AD
            7 = test_BC
            8 = test_BD
            9 = test_CD

    Event-code logic:
        Training codes:
            S111, S112, S113, S114 = yellow congruent A/B/C/D
            S121, S122, S123, S124 = yellow incongruent A/B/C/D
            S211, S212, S213, S214 = blue congruent A/B/C/D
            S221, S222, S223, S224 = blue incongruent A/B/C/D

        Test codes:
            S12 / S21 = AB
            S13 / S31 = AC
            S14 / S41 = AD
            S23 / S32 = BC
            S24 / S42 = BD
            S34 / S43 = CD
    """

    ev = pd.read_csv(events_tsv_path, sep="\t").copy()
    ev = ev.reset_index(drop=True)

    required_cols = ["onset", "trial_type", "value"]
    for col in required_cols:
        if col not in ev.columns:
            raise ValueError(
                f"Column '{col}' not found in {events_tsv_path}. "
                f"Available columns: {list(ev.columns)}"
            )

    ev["trial_type_str"] = ev["trial_type"].astype(str).str.strip()
    ev["value_str"] = ev["value"].astype(str).str.strip()

    # Extract numeric marker from values such as "S114", "S 21", "S  9".
    ev["value_num"] = (
        ev["value_str"]
        .str.extract(r"S\s*(\d+)", expand=False)
    )
    ev["value_num"] = pd.to_numeric(ev["value_num"], errors="coerce")

    # Use both training and test stimulus events.
    stim_mask = ev["trial_type_str"].str.startswith(("Trn Stim", "Train Stim", "Test Stim", "Tst Stim"))
    s2 = ev[stim_mask].copy().reset_index(drop=True)

    def map_stimulus_type(row):
        trial_type = str(row["trial_type_str"])
        v = row["value_num"]

        if pd.isna(v):
            return None

        v = int(v)

        # -------------------------
        # Training phase
        # -------------------------
        if trial_type.startswith(("Trn Stim", "Train Stim")):
            # Three-digit code: color / congruency / stimulus identity
            # 1xx = yellow, 2xx = blue
            # x1x = congruent, x2x = incongruent
            color = v // 100
            congruency = (v // 10) % 10

            if color == 1 and congruency == 1:
                return 0  # train_yellow_congruent
            elif color == 1 and congruency == 2:
                return 1  # train_yellow_incongruent
            elif color == 2 and congruency == 1:
                return 2  # train_blue_congruent
            elif color == 2 and congruency == 2:
                return 3  # train_blue_incongruent
            else:
                return None

        # -------------------------
        # Test phase
        # -------------------------
        elif trial_type.startswith(("Test Stim", "Tst Stim")):
            # Two-digit ordered pair, collapsed to unordered pair.
            # For example, S12 and S21 are both AB.
            first = v // 10
            second = v % 10

            if first == second:
                return None

            pair = tuple(sorted([first, second]))

            pair_to_label = {
                (1, 2): 4,  # AB
                (1, 3): 5,  # AC
                (1, 4): 6,  # AD
                (2, 3): 7,  # BC
                (2, 4): 8,  # BD
                (3, 4): 9,  # CD
            }

            return pair_to_label.get(pair, None)

        else:
            return None

    s2["stimulus_type"] = s2.apply(map_stimulus_type, axis=1)

    s2 = s2.dropna(subset=["onset", "stimulus_type"]).copy()
    s2["stimulus_type"] = s2["stimulus_type"].astype(int)

    if len(s2) == 0:
        print("Available trial_type values:")
        print(ev["trial_type"].value_counts(dropna=False).head(50))

        print("Available event values:")
        print(ev["value"].value_counts(dropna=False).head(50))

        raise ValueError(
            f"No valid training/test stimulus events found in {events_tsv_path}."
        )

    # Construct cue-locked events using onset times from events.tsv.
    sfreq = raw.info["sfreq"]
    print(f"Current sampling frequency: {sfreq} Hz")

    samples = np.round(
        s2["onset"].values.astype(float) * sfreq
    ).astype(int)

    events = np.c_[
        samples,
        np.zeros_like(samples, dtype=int),
        np.ones_like(samples, dtype=int),
    ]

    picks = mne.pick_types(
        raw.info,
        eeg=True,
        eog=False,
        exclude="bads",
    )

    epochs = mne.Epochs(
        raw,
        events,
        event_id={"cue": 1},
        tmin=tmin,
        tmax=tmax,
        baseline=baseline,
        picks=picks,
        proj=False,
        preload=True,
        reject=None,
        verbose=False,
    )

    data = epochs.get_data()  # (N, C, T)
    target_len = int(round((tmax - tmin) * sfreq))

    if data.shape[-1] > target_len:
        data = data[..., :target_len]

    elif data.shape[-1] < target_len:
        pad = target_len - data.shape[-1]
        data = np.pad(
            data,
            ((0, 0), (0, 0), (0, pad)),
            mode="edge",
        )

    X = np.transpose(data, (0, 2, 1))  # (N, T, C)

    # Align labels if MNE drops out-of-bound epochs.
    stimulus_type = s2["stimulus_type"].values.astype(int)
    stimulus_type = stimulus_type[epochs.selection]

    y = np.column_stack([
        np.full_like(stimulus_type, disease_id),
        stimulus_type,
        np.full_like(stimulus_type, subject_id),
        np.full_like(stimulus_type, task_id),
    ])

    assert X.shape[0] == y.shape[0], (
        f"X/y mismatch: X={X.shape}, y={y.shape}"
    )

    label_names = {
        0: "train_yellow_congruent",
        1: "train_yellow_incongruent",
        2: "train_blue_congruent",
        3: "train_blue_incongruent",
        4: "test_AB",
        5: "test_AC",
        6: "test_AD",
        7: "test_BC",
        8: "test_BD",
        9: "test_CD",
    }

    print("Task: Simon Conflict training/test stimulus classification")
    print(f"Number of retained trials: {len(stimulus_type)}")
    print("Label distribution:")

    labels, counts = np.unique(stimulus_type, return_counts=True)

    for label, count in zip(labels, counts):
        print(f"  {label}: {label_names[label]} = {count}")

    return X, y

## PASS 1: Find total number of trials across all subjects

In [24]:
N_total = 0  # total number of trials across all subjects/sessions, used to pre-allocate memmap

for sub in os.listdir(root):
    if 'sub-' in sub:
        # Skip sub-026 here
        if sub == "sub-026":
            print("❌ Skipping subject sub-026 due to ICA explained variance issue.")
            continue

        sub_path = os.path.join(root, sub)
        for ses in os.listdir(sub_path):
            if 'ses-' in ses:
                print(f"Current subject/session: {sub} / {ses}")
                eeg_path = os.path.join(sub_path, ses, 'eeg/')

                # Initialize file paths
                set_file_path, events_file_path = None, None

                # Traverse the eeg folder to find files
                for file in os.listdir(eeg_path):
                    if file.endswith('.set'):
                        set_file_path = os.path.join(eeg_path, file)
                    if 'events.tsv' in file:
                        events_file_path = os.path.join(eeg_path, file)

                # Error checking
                if set_file_path is None:
                    raise FileNotFoundError(f".set file not found in: {eeg_path}")
                if events_file_path is None:
                    raise FileNotFoundError(f"events.tsv not found in: {eeg_path}")

                # Load EEG data and preprocess
                print("Start preprocessing EEG data...")
                raw = mne.io.read_raw_eeglab(set_file_path, preload=True)
                # raw = data_preprocessing(raw, common_channels, SAMPLE_RATE, notch_freq=60, l_freq=0.5, h_freq=40, verbose=True)
                print()


                # # Extract subject_id and disease_id

                subject_id = int(sub.split('-')[-1])
                disease_id = participants[participants['participant_id'] == sub]['Group'].values[0]
                if disease_id == 'CTL':
                    disease_id = 0
                elif disease_id == 'PD':
                    disease_id = 1

                print(f"Subject ID: {subject_id}, Disease ID: {disease_id}")
                print("Start epoching and making X, y...")

                # Remove beh_file_path, keep other parameters unchanged
                X, y = epoch_and_make_xy(
                    raw, events_file_path,
                    tmin=-0.5, tmax=1.0, baseline=(-0.3, -0.2),   # according to paper
                    task_id=2, subject_id=subject_id, disease_id=disease_id
                )  # task_id=2 for Simon Conflict

                print(f"X shape: {X.shape}, y shape: {y.shape}")  
                # X: (N, SAMPLE_RATE * (tmax - tmin), C)
                # y: (N, 4) = [disease_id, stimulus_type, subject_id, task_id]

                N_total += X.shape[0]

                print("------------------------------------------------\n")

Current subject/session: sub-001 / ses-01
Start preprocessing EEG data...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\SCPD\SCPD\sub-001\ses-01\eeg\sub-001_ses-01_task-SimonConflict_eeg.fdt
Reading 0 ... 1256924  =      0.000 ...  2513.848 secs...

Subject ID: 1, Disease ID: 0
Start epoching and making X, y...
Current sampling frequency: 500.0 Hz
Task: Simon Conflict training/test stimulus classification
Number of retained trials: 561
Label distribution:
  0: train_yellow_congruent = 94
  1: train_yellow_incongruent = 93
  2: train_blue_congruent = 86
  3: train_blue_incongruent = 96
  4: test_AB = 32
  5: test_AC = 32
  6: test_AD = 32
  7: test_BC = 32
  8: test_BD = 32
  9: test_CD = 32
X shape: (561, 750, 67), y shape: (561, 4)
------------------------------------------------

Current subject/session: sub-002 / ses-01
Start preprocessing EEG data...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\SCPD\SCPD\sub-002\ses-01\eeg\sub-002_ses-01_task-SimonConflict_

In [27]:
output_root = os.path.join('Processed', sub_folder_path, 'SCPD')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
SEG_LEN = int((1.0 - (-0.5)) * SAMPLE_RATE)  # samples per epoch
n_channels = len(common_channels)
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 4))

X path: Processed\200Hz\SCPD\X.dat
y path: Processed\200Hz\SCPD\y.dat


## PASS 2: Process and store trials into memmap

In [28]:
cur = 0  # current index in memmap

# Iterate over subjects/sessions 
for sub in os.listdir(root):
    if 'sub-' in sub:
        # Skip sub-026 here
        if sub == "sub-026":
            print("❌ Skipping subject sub-026 due to ICA explained variance issue.")
            continue

        sub_path = os.path.join(root, sub)
        for ses in os.listdir(sub_path):
            if 'ses-' in ses:
                print(f"Current subject/session: {sub} / {ses}")
                eeg_path = os.path.join(sub_path, ses, 'eeg/')

                # Initialize file paths
                set_file_path, events_file_path = None, None

                # Traverse the eeg folder to find files
                for file in os.listdir(eeg_path):
                    if file.endswith('.set'):
                        set_file_path = os.path.join(eeg_path, file)
                    if 'events.tsv' in file:
                        events_file_path = os.path.join(eeg_path, file)

                # Error checking
                if set_file_path is None:
                    raise FileNotFoundError(f".set file not found in: {eeg_path}")
                if events_file_path is None:
                    raise FileNotFoundError(f"events.tsv not found in: {eeg_path}")

                # Load EEG data and preprocess
                print("Start preprocessing EEG data...")
                raw = mne.io.read_raw_eeglab(set_file_path, preload=True)
                raw = data_preprocessing(raw, common_channels, SAMPLE_RATE, notch_freq=60, l_freq=0.5, h_freq=40, verbose=True)
                print()


                # # Extract subject_id and disease_id

                subject_id = int(sub.split('-')[-1])
                disease_id = participants[participants['participant_id'] == sub]['Group'].values[0]
                if disease_id == 'CTL':
                    disease_id = 0
                elif disease_id == 'PD':
                    disease_id = 1

                print(f"Subject ID: {subject_id}, Disease ID: {disease_id}")
                print("Start epoching and making X, y...")

                # Remove beh_file_path, keep other parameters unchanged
                X, y = epoch_and_make_xy(
                    raw, events_file_path,
                    tmin=-0.5, tmax=1.0, baseline=(-0.3, -0.2),   # according to paper
                    task_id=2, subject_id=subject_id, disease_id=disease_id
                )  # task_id=2 for Simon Conflict

                print(f"X shape: {X.shape}, y shape: {y.shape}")  
                # X: (N, SAMPLE_RATE * (tmax - tmin), C)
                # y: (N, 4) = [disease_id, stimulus_type, subject_id, task_id]

                # Save to memmap
                X_mm[cur:cur+X.shape[0]] = X
                y_mm[cur:cur+y.shape[0]] = y
                cur += X.shape[0]

                print("------------------------------------------------\n")
                
print("DONE: cur =", cur, " expected N_total =", N_total)

# flush
del X_mm
del y_mm  

Current subject/session: sub-001 / ses-01
Start preprocessing EEG data...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\SCPD\SCPD\sub-001\ses-01\eeg\sub-001_ses-01_task-SimonConflict_eeg.fdt
Reading 0 ... 1256924  =      0.000 ...  2513.848 secs...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
✔ Step 1: Picked & reordered channels ['FC2', 'P6', 'FT8', 'FC5', 'CP1', 'F4', 'AF4', 'O1', 'C4', 'CP6', 'P7', 'AF8', 'PO8', 'AF7', 'TP7', 'F7', 'Pz', 'FC4', 'CP5', 'Fp2', 'P1', 'POz', 'Fp1', 'CP3', 'CP2', 'F8', 'PO4', 'PO7', 'P8', 'P5', 'AFz', 'T8', 'Cz', 'FC1', 'F1', 'PO3', 'C2', 'P2', 'Fz', 'Oz', 'C3', 'P4', 'FC6', 'P3', 'FC3', 'C1', 'C5', 'C6', 'F6', 'FT7', 'O2', 'CP4', 'F3', 'F2', 'F5', 'AF3', 'TP8', 'T7', 'FCz']
✔ Step 2: Montage set to 'standard_1020'.
✔ Step 3: Notch filtered at 60 Hz
✔ Step 4: Band-pass filtered from 0.5–40 Hz
ℹ Step 5: No bad channels to interpolate.
✔ Step 6: EEG re-referenced to average
✔ Step 8: Resampled to 200 Hz

Subje

## Create JSON file

In [29]:
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE": SAMPLE_RATE,
    "TMIN": -0.5,
    "TMAX": 1.0,
    "BASELINE": [-0.3, -0.2],
    "MONTAGE": "standard_1020",
    "CHANNELS": common_channels,
    "LABELS": {
        "disease_id": {0: "CTL", 1: "PD"},
        "stimulus_type": {0: "train_yellow_congruent", 1: "train_yellow_incongruent", 2: "train_blue_congruent", 3: "train_blue_incongruent", 4: "test_AB", 5: "test_AC", 6: "test_AD", 7: "test_BC", 8: "test_BD", 9: "test_CD"},
        "subject_id": "integer from 1 to number of subjects",
        "task_id": {2: "Simon Conflict"}
    }
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

Saved meta: Processed\200Hz\SCPD\meta.json


## Load and check the processed data

In [30]:
# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]

print("Meta:")
for k, v in meta.items():
    print(f"{k}: {v}")
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 4))

# subject_id is stored in y[:, 2]
subject_ids = np.unique(y_mm[:, 2]).astype(int)

total_trials = 0
for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 2] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 4)       # logical y shape for this subject
    total_trials += n_seg
    
    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

print("\nTotal subjects:", len(subject_ids))
print("Total trials:", total_trials)
# option: delete memmap views
del X_mm, y_mm

Meta:
N: 46193
T: 300
C: 59
SAMPLE_RATE: 200
TMIN: -0.5
TMAX: 1.0
BASELINE: [-0.3, -0.2]
MONTAGE: standard_1020
CHANNELS: ['FC2', 'P6', 'FT8', 'FC5', 'CP1', 'F4', 'AF4', 'O1', 'C4', 'CP6', 'P7', 'AF8', 'PO8', 'AF7', 'TP7', 'F7', 'Pz', 'FC4', 'CP5', 'Fp2', 'P1', 'POz', 'Fp1', 'CP3', 'CP2', 'F8', 'PO4', 'PO7', 'P8', 'P5', 'AFz', 'T8', 'Cz', 'FC1', 'F1', 'PO3', 'C2', 'P2', 'Fz', 'Oz', 'C3', 'P4', 'FC6', 'P3', 'FC3', 'C1', 'C5', 'C6', 'F6', 'FT7', 'O2', 'CP4', 'F3', 'F2', 'F5', 'AF3', 'TP8', 'T7', 'FCz']
LABELS: {'disease_id': {'0': 'CTL', '1': 'PD'}, 'stimulus_type': {'0': 'train_yellow_congruent', '1': 'train_yellow_incongruent', '2': 'train_blue_congruent', '3': 'train_blue_incongruent', '4': 'test_AB', '5': 'test_AC', '6': 'test_AD', '7': 'test_BC', '8': 'test_BD', '9': 'test_CD'}, 'subject_id': 'integer from 1 to number of subjects', 'task_id': {'2': 'Simon Conflict'}}
-------------------------------------
Subject ID 001: X shape=(561, 300, 59), y shape=(561, 4)
Subject ID 002: X shap